# Synthea Stories: generate GP notes and patient letters

This notebook turns a synthetic patient record from [Synthea](https://synthea.mitre.org/) (a C-CDA `.xml` file)
into a long, realistic text record: GP consultation notes and messages from the patient, written by an
open-weight LLM running on Colab's free GPU. **Every patient and document is fictional.**

**Before you start:** choose *Runtime → Change runtime type → T4 GPU*, then run the cells in order.

How long it takes: setup is about 5 minutes. On the free T4 GPU, each document takes very roughly 20–60 seconds,
so a patient with ~45 documents takes about 15–45 minutes.

**If Colab disconnects:** run steps 1–5 again, then re-run the generation step. Documents already written are
saved in Google Drive and are skipped, so it carries on where it stopped.

## 1. Check the GPU
You should see a Tesla T4 below. If you see an error, change the runtime type as described above.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Get the code

In [ ]:
import os, sys
REPO_DIR = "/content/Synthea-Stories"
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q https://github.com/Seth-Bullock/Synthea-Stories.git {REPO_DIR}
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("Code is in", REPO_DIR)

## 3. Choose where to save the output
Saving to Google Drive (recommended) means nothing is lost if Colab disconnects. You will be asked to allow
access to your Drive. Set `USE_DRIVE = False` to save inside this Colab session instead (deleted when it ends).

In [ ]:
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = "/content/drive/MyDrive/synthea-stories-output"
else:
    OUTPUT_DIR = "/content/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Output will be saved in", OUTPUT_DIR)

## 4. Install and start Ollama
[Ollama](https://ollama.com) runs open-weight LLMs locally on the GPU.

In [ ]:
import subprocess, time, urllib.request

!sudo apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null

ollama = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT,
                          env={**os.environ, "OLLAMA_KEEP_ALIVE": "-1"})
for _ in range(30):
    try:
        urllib.request.urlopen("http://localhost:11434")
        print("Ollama is running.")
        break
    except Exception:
        time.sleep(1)
else:
    print("Ollama did not start; see /content/ollama.log")

## 5. Download the model
`gemma3:12b` (Google's Gemma 3, 12 billion parameters, about 8 GB) is the largest good writer that fits
comfortably on the free T4 GPU. For a faster but weaker model, try `gemma3:4b`. Other options include
`qwen2.5:14b` and `llama3.1:8b`.

In [ ]:
MODEL = "gemma3:12b"
!ollama pull {MODEL} 2>&1 | tail -1

from gpnotes.llm import OllamaClient
client = OllamaClient(model=MODEL, num_ctx=8192, temperature=0.8)
print(client.chat("You are a helpful assistant.", "Say hello in exactly five words."))

## 6. Choose the patient record(s)
By default this uses the sample patient that comes with the code. To use your own Synthea C-CDA files,
set `UPLOAD = True` and choose one or more `.xml` files when prompted.

In [ ]:
UPLOAD = False

if UPLOAD:
    from google.colab import files
    os.makedirs("/content/input", exist_ok=True)
    for name, data in files.upload().items():
        with open(os.path.join("/content/input", name), "wb") as f:
            f.write(data)
    CCDA_FILES = sorted(os.path.join("/content/input", n) for n in os.listdir("/content/input") if n.endswith(".xml"))
else:
    CCDA_FILES = ["samples/Alexandra16_Mosciski958.xml"]
print(CCDA_FILES)

## 7. Quick test: the first few episodes of one patient
This writes the first 6 episodes (about 8 documents) to a separate `test` folder so you can check the style
before committing to a full run. The exact prompts sent to the model are saved in `<patient>/prompts/`.

In [ ]:
from gpnotes.generate import generate_patient

test_file = generate_patient(CCDA_FILES[0], client, out_dir=os.path.join(OUTPUT_DIR, "test"), max_episodes=6)
print(open(test_file).read())

## 8. Full run
Generates the complete record for every chosen patient. Safe to re-run after a disconnection.

In [ ]:
outputs = []
for i, path in enumerate(CCDA_FILES, start=1):
    print(f"\n=== Patient {i} of {len(CCDA_FILES)}: {os.path.basename(path)}")
    outputs.append(generate_patient(path, client, out_dir=OUTPUT_DIR))

## 9. Read and download the results

In [ ]:
from google.colab import files

print(open(outputs[0]).read()[:5000])
for path in outputs:
    files.download(str(path))